<a href="https://colab.research.google.com/github/chetnapriyadarshini/NMT_Encoder_Decoder_Attention_Beam/blob/main/nmt_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.model_selection import train_test_split

import unicodedata
import re
import numpy as np
import os
import io
import time
#!pip3 install indic-nlp-library

In [3]:
!pip3 install indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.7 MB/s eta 0:00:00


In [2]:
!wget "http://lotus.kuee.kyoto-u.ac.jp/WAT/indic-multilingual/indic_languages_corpus.tar.gz"

--2026-03-07 14:45:48--  http://lotus.kuee.kyoto-u.ac.jp/WAT/indic-multilingual/indic_languages_corpus.tar.gz
Resolving lotus.kuee.kyoto-u.ac.jp (lotus.kuee.kyoto-u.ac.jp)... 130.54.208.131
Connecting to lotus.kuee.kyoto-u.ac.jp (lotus.kuee.kyoto-u.ac.jp)|130.54.208.131|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132762852 (127M) [application/x-gzip]
Saving to: ‘indic_languages_corpus.tar.gz’

indic_languages_cor 100%[===================>] 126.61M  29.6MB/s    in 4.9s    

2026-03-07 14:45:54 (25.8 MB/s) - ‘indic_languages_corpus.tar.gz’ saved [132762852/132762852]



In [3]:
import tarfile
with tarfile.open("indic_languages_corpus.tar.gz", "r:gz") as tar:
  tar.extractall()
print("Done")

/tmp/ipykernel_452/2267547997.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Done


In [4]:
#We copy the Hindi to English files for working in this example (dev.en, dev.hi, test.en, test.hi and train.en, train.hi)
%cp indic_languages_corpus/bilingual/hi-en/* .
#Clean up to avoid storing these files in the session
%rm -r indic_languages_corpus indic_languages_corpus.tar.gz

In [5]:
# Install Nirmala font which we would need for plotting later in the code
!wget "https://www.wfonts.com/download/data/2016/04/29/nirmala-ui/nirmala-ui.zip"

--2026-03-07 14:46:34--  https://www.wfonts.com/download/data/2016/04/29/nirmala-ui/nirmala-ui.zip
Resolving www.wfonts.com (www.wfonts.com)... 104.21.1.127, 172.67.129.58, 2606:4700:3031::ac43:813a, ...
Connecting to www.wfonts.com (www.wfonts.com)|104.21.1.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2677244 (2.6M) [application/x-zip-compressed]
Saving to: ‘nirmala-ui.zip’

nirmala-ui.zip      100%[===================>]   2.55M  16.4MB/s    in 0.2s    

2026-03-07 14:46:35 (16.4 MB/s) - ‘nirmala-ui.zip’ saved [2677244/2677244]



In [6]:
from zipfile import ZipFile
with ZipFile('nirmala-ui.zip', 'r') as zipObj:
   # Extract all the contents of zip file in current directory
   zipObj.extractall()
print("done!")

done!


In [7]:
%rm nirmala-ui.* sharefonts.* nirmala.png

rm: cannot remove 'sharefonts.*': No such file or directory


In [8]:
f = open("train.hi")
w1 = f.readlines()
print(len(w1))
print(w1[0:5])
g = open("train.en")
w2 = g.readlines()
print(len(w2))
print(w2[0:5])

84557
['और उनके Sigil क्या है?\n', 'मैं मरना नहीं चाहता.\n', 'यह मुझे लगता है कि एक ही देश है.\n', 'फिर ये नन्हें बच्चों की तरह रोएँगे।\n', 'नहीं, मुझे पावर की जरुरत है !\n']
84557
['And what is their Sigil?\n', 'I do not want to die.\n', "It's the same country I think.\n", "Then they'll be crying like babies.\n", '- No, I need power up!\n']


In [9]:
# Restrict the total number of sentences to 70000
NUM_SENTENCES = 70000

In [10]:
# strip the input and output of extra unnecessary characters
# store all the cleaned input and output sentences into input_sentences[] and output_sentences[]
# tokenize the Hindi (target) sentences using the indicNLP libary class and add <sos> (start-of-sentence) and <eos> (end-of-sentence)

input_sentences = []
output_sentences = []

count = 0
for line in open(r'train.en', encoding="utf-8"):
    count += 1

    if count > NUM_SENTENCES:
        break

    input_sentence = line.rstrip().strip("\n").strip('-') #we strip the sentence of '\n' and '-'
    input_sentences.append(input_sentence) #store all input sentences in the input sentences list

count = 0

for line in open(r'train.hi'):
    count += 1

    if count > NUM_SENTENCES:
        break
    output_sentence =  line.rstrip().strip("\n").strip('-')
    from indicnlp.tokenize import indic_tokenize
    line = indic_tokenize.trivial_tokenize(output_sentence) #we tokenize the hindi sentences

    output_sentences.append(['<sos>'] + line + ['<eos>']) #append the start and end tags to the tokenised sentences
                                                          #each tokenied sentence is stored as a list in output sentences
print(type(input_sentences[10]))
print(type(output_sentences[10]))

<class 'str'>
<class 'list'>


In [11]:
print(input_sentences[-1])
print(output_sentences[-1])

Her face.
['<sos>', 'उसका', 'चेहरा', '.', '<eos>']


In [12]:
def unicode_to_ascii(s):
  return ''.join(c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn')


def preprocess_sentence(w):
  w = unicode_to_ascii(w.lower().strip())

  # creating a space between a word and the punctuation following it
  # eg: "he is a boy." => "he is a boy ."

  w = re.sub(r"([?.!,¿])", r" \1 ", w)
  w = re.sub(r'[" "]+', " ", w)

  # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
  w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)

  w = w.strip()

  # adding a start and an end token to the sentence
  # so that the model know when to start and stop predicting.
  w = '<sos> ' + w + ' <eos>'
  return w

In [13]:
for i in range(len(input_sentences)):
   input_sentences[i] = preprocess_sentence(input_sentences[i])

print(input_sentences[8])
print(output_sentences[8])

<sos> i told her we rest on sundays . <eos>
['<sos>', 'मैं', 'रविवार', 'को', 'उसे', 'हम', 'बाकी', 'बताया', '.', '<eos>']


In [14]:
def sample_function(lang):
  lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")
  lang_tokenizer.fit_on_texts(lang)

  tensor = lang_tokenizer.texts_to_sequences(lang)
  tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')

  return tensor, lang_tokenizer

In [15]:
def load_dataset(input_lang, target_lang):
  input_tensor, inp_lang_tokenizer = sample_function(input_lang)
  target_tensor, target_lang_tokenizer = sample_function(target_lang)
  return input_tensor, inp_lang_tokenizer, target_tensor, target_lang_tokenizer

In [16]:
inp_tensor, inp_lang, target_tensor, target_lang = load_dataset(input_sentences, output_sentences)

max_length_input, max_length_target = inp_tensor.shape[1], target_tensor.shape[1]

In [17]:
print(max_length_input, max_length_target)

72 69


In [18]:
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(inp_tensor, target_tensor)

In [19]:
print(len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val))

52500 52500 17500 17500


In [20]:
def convert(lang, tensor):
  for t in tensor:
    if t != 0:
      print("%d----------%s", (t, lang.index_word[t]))

In [21]:
print ("Input Language; index to word mapping")
convert(inp_lang, input_tensor_train[0])
print ()
print ("Target Language; index to word mapping")
convert(target_lang, target_tensor_train[0])

Input Language; index to word mapping
%d----------%s (np.int32(1), '<sos>')
%d----------%s (np.int32(91), 'where')
%d----------%s (np.int32(19), 'is')
%d----------%s (np.int32(60), 'she')
%d----------%s (np.int32(8), '?')
%d----------%s (np.int32(91), 'where')
%d----------%s (np.int32(10), 's')
%d----------%s (np.int32(899), 'san')
%d----------%s (np.int32(8), '?')
%d----------%s (np.int32(2), '<eos>')

Target Language; index to word mapping
%d----------%s (np.int32(1), '<sos>')
%d----------%s (np.int32(952), 'सैन')
%d----------%s (np.int32(366), 'कहां')
%d----------%s (np.int32(4), 'है')
%d----------%s (np.int32(8), '?')
%d----------%s (np.int32(2), '<eos>')


In [22]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64
steps_per_epoch = BUFFER_SIZE//BATCH_SIZE
EMBEDDING_DIM = 256
UNITS = 1024 #number of gated recurrent units

vocab_inp_size = len(inp_lang.word_index)+1
vocab_tar_size = len(target_lang.word_index)+1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

print(BUFFER_SIZE)
print(steps_per_epoch)
print(vocab_inp_size)
print(vocab_tar_size)

52500
820
17619
22224


In [23]:
example_input_batch, example_target_batch = next(iter(dataset))
example_input_batch.shape, example_target_batch.shape

(TensorShape([64, 72]), TensorShape([64, 69]))

# **Encoder-Decoder model with attention**

The encoder model consists of an embedding layer, a GRU layer with 1024 units.

The decoder model consists of an attention layer, a embedding layer, a GRU layer and a dense layer.

The attention model consists of three dense layers (BahdanauAttention Model) .


---
![picture](https://drive.google.com/uc?id=1AnbdmNzOi9WyEZ8RiMWL3MsndVliggs7)

In [24]:
class Encoder(tf.keras.Model):

  def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
    super(Encoder, self).__init__()
    self.batch_sz = batch_sz
    self.enc_units = enc_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')


  def call(self, x, hidden_state):
    x = self.embedding(x)
    output, hidden = self.gru(x, initial_state=hidden_state)
    return output, hidden

  def initialize_hidden_state(self):
    # Fix: tf.zeros expects the shape as a tuple/list in its first argument
    return tf.zeros((self.batch_sz, self.enc_units))

encoder = Encoder(vocab_inp_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)

sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)

In [25]:
print ('Encoder output shape: (batch size, sequence length, units) {}'.format(sample_output.shape))
print ('Encoder Hidden state shape: (batch size, units) {}'.format(sample_hidden.shape))

Encoder output shape: (batch size, sequence length, units) (64, 72, 1024)
Encoder Hidden state shape: (batch size, units) (64, 1024)


The Bahdanau Attention layer takes 2 inputs:

* Decoder's hidden state: represented by **Query**
* Encoder's Output: represented by **Value**

In [26]:
class BahdanauAttention(tf.keras.layers.Layer):
  def __init__(self, units):
    super(BahdanauAttention, self).__init__()
    self.W1 = tf.keras.layers.Dense(units)
    self.W2 = tf.keras.layers.Dense(units)
    self.V = tf.keras.layers.Dense(1)

  def call(self, query, values): #query is decoder hidden state, values is encoder output #values = batch_size, max_len, hidden_size
    #64, 72, 1024
    query_with_time_axis = tf.expand_dims(query, 1) #query = decoder_hidden_state = batch_size, hidden_size = 64, 1024, o/p= 64,1,1024
    score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values))) #additive attention #[64,1,1024] + [64,72,1024] = V[64,72,1024] = [64,72,1]
    attention_weights = tf.nn.softmax(score, axis=1) #batch_size, max_length, 1 [64,72,1]
    context_vector = attention_weights * values # [64,72,1] * [64,72,1024] = [64,72,1024]
    context_vector = tf.reduce_sum(context_vector, axis=1) #[64,1024]
    return context_vector, attention_weights


In [27]:
class Decoder(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
    super(Decoder, self).__init__()
    self.batch_sz = batch_sz #64
    self.dec_units = dec_units #1024
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)#17619, 256
    self.attention = BahdanauAttention(self.dec_units)
    self.gru = tf.keras.layers.GRU(self.dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
    self.fc = tf.keras.layers.Dense(vocab_size)

  def call(self, x, hidden, enc_output):
    context_vector, attention_weights = self.attention(hidden, enc_output) #[64,1024], [64,72,1]
    # x shape after passing through embedding == (batch_size, 1, embedding_dim)
    x = self.embedding(x) #[64,1,256]
     # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1) #[64,1,1024+256] = #[64,1,1280]
    output, state = self.gru(x) #[64,1,1024], [64,1024]
     # output shape == (batch_size * 1, hidden_size)
    output = tf.reshape(output, (-1, output.shape[2])) #[64,1024]
    x = self.fc(output) #[64,17619]
    return x, state, attention_weights



In [28]:
decoder = Decoder(vocab_tar_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)

sample_decoder_output, _, _ = decoder(tf.random.uniform((BATCH_SIZE, 1)), sample_hidden, sample_output)

print ('Decoder output shape: (batch_size, vocab size) {}'.format(sample_decoder_output.shape))


Decoder output shape: (batch_size, vocab size) (64, 22224)


In [29]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none') #Loss function is categorical crossentropy

def loss_function(real, pred):
  mask = tf.math.logical_not(tf.math.equal(real, 0))
  loss_ = loss_object(real, pred)

  mask = tf.cast(mask, dtype=loss_.dtype)
  loss_ *= mask

  return tf.reduce_mean(loss_)

In [30]:
checkpoint_dir = './tutorial_checkpoint'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 encoder=encoder,
                                 decoder=decoder)

In [31]:
@tf.function
def train_step(inp, targ, enc_hidden):
  loss = 0

  with tf.GradientTape() as tape:

    enc_output, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([target_lang.word_index['<sos>']] * BATCH_SIZE, 1)

    for t in range(1, targ.shape[1]):
      predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
      loss += loss_function(targ[:, t], predictions)
      #teacher forcing
      dec_input = tf.expand_dims(targ[:,t], 1)

    batch_loss += (loss / int(targ.shape[1]))
    total_loss += batch_loss

    variables = encoder.variables + decoder.variables

    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss

In [32]:
train = False
EPOCHS = 15

if train:
  for epoch in range(EPOCHS):
    start = time.time
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    for batch, (inp, targ) in enumerate(dataset.take(steps_per_epoch)):
      batch_loss = train_step(inp, targ, enc_hidden)
      total_loss += batch_loss

      if batch % 100 == 0:
        print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1, batch, batch_loss.numpy()))

      if epoch+1 % 2 == 0:
        checkpoint.save(file_prefix = checkpoint_prefix)

      print('Epoch {} Loss {:.4f}'.format(epoch + 1,
                                        total_loss / steps_per_epoch))
      print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))


In [1]:
def evaluate(sentence):
  attention_plot = np.zeros((max_length_target, max_length_input))

  sentence = preprocess_sentence(sentence)

  inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
  inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_input, padding='post')

  inputs = tf.convert_to_tensor(inputs)

  result = ""

  hidden = [tf.zeros((1, units))]

  enc_out, enc_hidden = encoder(inputs, hidden)

  dec_hidden = enc_hidden
  dec_input = tf.expand_dims([target_lang.word_index['<sos>']], 0)

  for t in range(max_length_targ):
    predictions, dec_hidden, attention_weights = decoder(dec_input,
                                                          dec_hidden,
                                                          enc_out)

    # pass the encoder output, decoder hidden state(which is initialised to encoder hidden state for the first time and decoder input to the decoder)
    # make a prediction and obtain decoder hidden states and attention weights

    # storing the attention weights to plot later on
    attention_weights = tf.reshape(attention_weights, (-1, ))
    attention_plot[t] = attention_weights.numpy()

    predicted_id = tf.argmax(predictions[0]).numpy()

    result += targ_lang.index_word[predicted_id] + ' '

    if targ_lang.index_word[predicted_id] == '<eos>':
      return result, sentence, attention_plot

    # the predicted ID is fed back into the model
    dec_input = tf.expand_dims([predicted_id], 0)

  return result, sentence, attention_plot

In [ ]:
# function for plotting the attention weights
from matplotlib.font_manager import FontProperties

def plot_attention(attention, sentence, predicted_sentence):
  """
  This defines the function that takes three arguments:

  attention: A 2D NumPy array representing the attention weights (e.g., max_length_target rows by max_length_input columns).
  sentence: The preprocessed input sentence (English).
  predicted_sentence: The output sentence generated by the model (Hindi).
  """
  #Creates a new figure for the plot with a specified size of 10x10 inches.
  fig = plt.figure(figsize=(10,10))
  #Adds a single subplot to the figure. In this case, it means one plot occupying the entire figure.
  ax = fig.add_subplot(1, 1, 1)
  #It displays the attention matrix as an image:Creates a matrix plot, The 2D array of attention weights, where the intensity of the color at each point indicates the strength of the attention.
  #Specifies the colormap to use, viridis being a perceptually uniform colormap often used for scientific data.
  ax.matshow(attention, cmap='viridis')
  #Defines a dictionary for font properties, setting the font size to 18. This will be used for the English (input) sentence labels.
  fontdict = {'fontsize': 18}
  #loading the 'Nirmala.ttf' font file. This is crucial for correctly rendering Hindi characters on the y-axis, as default Matplotlib fonts might not support them.
  hindi_font = FontProperties(fname = 'Nirmala.ttf')
  """
  Sets the labels for the x-axis (horizontal).

  [''] + sentence: Prepends an empty string to the sentence list. This is a common Matplotlib trick to align the labels correctly with the matrix cells, as the matrix typically starts at index 0,0.
  fontdict=fontdict: Applies the defined font size to these labels.
  rotation=90: Rotates the labels by 90 degrees to prevent overlapping, especially with longer words.
  """
  ax.set_xticklabels([''] + sentence, fontdict=fontdict, rotation=90)
  """
  Sets the labels for the y-axis (vertical).

  [''] + predicted_sentence: Similar to the x-axis, prepends an empty string for alignment.
  fontproperties=hindi_font: Applies the custom Hindi font, ensuring correct display of the predicted Hindi words.
  fontsize='18': Sets the font size for these labels.
  ax.xaxis.set_major_locator(ticker.MultipleLocator(1)): Configures the x-axis ticks to appear at every integer interval (every 1 unit). This ensures a label for each input word.
  """
  ax.set_yticklabels([''] + predicted_sentence, fontproperties=hindi_font, fontsize='18')
  #Configures the x-axis ticks to appear at every integer interval (every 1 unit). This ensures a label for each input word.
  ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
  #Configures the y-axis ticks to appear at every integer interval (every 1 unit). This ensures a label for each output word.
  ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

  plt.show()

In [1]:
def translate(sentence, plotgraph):
  result, sentence, attention_plot = evaluate(sentence)
  print('Input: %s' % (sentence))
  print('Predicted translation: {}'.format(result))
  attention_plot = attention_plot[:len(result.split(' ')), :len(sentence.split(' '))]
  if plotgraph:
      plot_attention(attention_plot, sentence.split(' '), result.split(' '))

  return result

In [ ]:
test_input_sentences = []
test_output_sentences = []

for line in open(r'test.en', encoding="utf-8"):

    test_input_sentence = line.rstrip().strip("\n").strip('-')
    test_input_sentences.append(test_input_sentence)


for line in open(r'test.hi'):
    test_output_sentence =  line.rstrip().strip("\n").strip('-')
    line = indic_tokenize.trivial_tokenize(test_output_sentence)

    test_output_sentences.append(['<sos>'] + line + ['<eos>'])

print(type(test_input_sentences[90]))
print(len(test_output_sentences))
print(test_input_sentences[90])
print(test_output_sentences[90])

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.bleu_score import SmoothingFunction
chencherry = SmoothingFunction()
evaluate_n_sentences = 10

references = []
candidates = []
for i in range(evaluate_n_sentences):
  try:
    res = translate(test_input_sentences[i], False)
    ref = test_output_sentences[i].copy()
    ref = [e for e in ref if e not in ('<eos>', '<sos>', '.')]
    references.append(ref)
    listToStr = ' '.join(map(str, test_output_sentences[i]))
    print('Reference Translation: %s' % (listToStr))
    candidate = indic_tokenize.trivial_tokenize(res)
    candidate = [e for e in candidate if e not in ('<', 'eos','>', '.')]
    candidates.append(candidate)
  except:
    print('Sentence :', i+1, ' not translatable ..moving to next' )
score1 = corpus_bleu(references, candidates, smoothing_function=chencherry.method4)
score2 = corpus_bleu(references, candidates)
print('BLEU score on test data without smoothing function: ' ,score2)
print('BLEU score on test data with smoothing function: ' ,score1)

# **Prediction using Beam Search**

We extend the Decoder Prediction Model to use the Beam Search algorithm with a configurable value of the Beam Width (k)

#### Beam Search Implementation

In [ ]:
def softmax(x):
    """Compute softmax values for each sets of scores in x.
    ubtracting the maximum value from x before exponentiation prevents very large numbers that could lead to overflow errors when calculating e^x.
    Then, e_x / e_x.sum() normalizes these exponentiated values so they sum up to 1, effectively turning them into probabilities.
    """
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def reduce_mul(l):
    """
    A simple utility function to calculate the product of all numbers in a list.
    In beam search, this is used to compute the cumulative probability (score) of a sequence by multiplying the probabilities
    of individual words in that sequence.
    """
    out = 1.0
    for x in l:
        out *= x
    return out

def check_all_done(seqs):
    """
    Purpose: This function checks if all active sequences in the beam have ended.
    A sequence is considered "done" if its last predicted token is the end-of-sentence (<eos>) token.
    How it works: It iterates through each sequence (seq) in the seqs list.
    If it finds any sequence where the last token (seq[-1][0]) is not <eos>, it immediately returns False.
     If it goes through all sequences and all of them end with <eos>, it returns True.
    """
    for seq in seqs:
        if (targ_lang.index_word[seq[-1][0]]!='<eos>') :
            return False
    return True

#seq: [[word,word],[word,word],[word,word]]
#output: [[word,word,word],[word,word,word],[word,word,word]]
def beam_search_step(dec_hid, enc_out, top_seqs, k):
  """
  Purpose: This is the core function that performs one step of the beam search. It takes the current top k candidate sequences,
  extends each by predicting the next possible words, and then selects the overall top k best new sequences.
  Inputs:
    dec_hid: The decoder's current hidden state.
    enc_out: The encoder's output (contextual information from the source sentence).
    top_seqs: A list of the k best sequences found so far from the previous step.
    Each sequence is represented as a list of (word_index, probability) tuples.
    k: The beam size (number of top sequences to keep).
  Detailed Steps:
    Initialization: all_seqs is an empty list that will store all possible extended sequences from the current step.
    Iterating through top_seqs: For each seq (candidate sequence) from the previous step:
    Calculate seq_score: It computes the cumulative probability of the current sequence seq using reduce_mul.
    Handle <eos>: If seq already ended with <eos>, it's added to all_seqs as a complete sequence,
    and no further prediction is made for it.
    Predict next word:
      dec_input: The last predicted word of the current seq is used as the input to the decoder.
      redictions, dec_hid, _ = decoder(...): The decoder generates logits for the next possible words and updates its hidden state.
      output_step = [...]: The softmax function is applied to the predictions to get probabilities for all possible next words.
                           These are paired with their word indices.
      output_step = sorted(...): These (index, prob) pairs are sorted in descending order of probability.
  Extend sequences: It takes the top k (or beam_size) words from output_step:
  For each of these top k words, a new score (score = seq_score * word_score) is calculated (cumulative probability).
  A new sequence rs_seq is formed by extending the current seq with this new word and its score.
  The done status is checked if the new word is <eos>.
  The (rs_seq, score, done) tuple is added to all_seqs.
  Select top k overall:
    all_seqs is sorted by their cumulative score in descending order.
    topk_seqs are the k best sequences from all_seqs.
    all_done = check_all_done(topk_seqs): It checks if all these k sequences have finished.
  Returns: The topk_seqs for the next step, the all_done flag, the updated dec_hid (which is typically just the dec_hid from
           the last decoder call within this step, potentially a simplification), and topk_scores
      """
    all_seqs = []
    scores=[]
    for seq in top_seqs:
        seq_score = reduce_mul([_score for _,_score in seq])
        if (targ_lang.index_word[seq[-1][0]]=='<eos>') :
            all_seqs.append((seq, seq_score, True))
            # Reached on end of string for this seq. Dont predict any further
            continue
        #get predictions using encoder_context & seq last element
        dec_input = tf.expand_dims([seq[-1][0]], 0)
        predictions, dec_hid, _ = decoder(dec_input, dec_hid, enc_out)
        output_step = [(idx,prob) for idx,prob in enumerate(softmax(predictions[0].numpy()))]
        output_step = sorted(output_step, key=lambda x: x[1], reverse=True)
        for i,word in enumerate(output_step):
            if i >= k:
                break
            word_index = word[0]
            word_score = word[1]
            score = seq_score * word_score
            rs_seq = seq + [word]
            done = (word_index == targ_lang.word_index['<eos>'])
            all_seqs.append((rs_seq, score, done))
    all_seqs = sorted(all_seqs, key = lambda seq: seq[1], reverse=True)
    topk_seqs = [seq for seq,_,_ in all_seqs[:k]]
    topk_scores = [scores for _,scores,_ in all_seqs[:k]]
    all_done = check_all_done(topk_seqs)
    return topk_seqs, all_done, dec_hid, topk_scores

def evaluate_beam_search(sentence):

  sentence = preprocess_sentence(sentence)

  inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
  inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],
                                                         maxlen=max_length_inp,
                                                         padding='post')
  inputs = tf.convert_to_tensor(inputs)

  hidden = [tf.zeros((1, units))]
  enc_out, enc_hidden = encoder(inputs, hidden)
  dec_hidden = enc_hidden
  beam_seqs = [[(targ_lang.word_index['<sos>'],1.0)]]
  beam_size = 3
  for _ in range(max_length_targ):
    beam_seqs, all_done, dec_hidden, scores_seq = beam_search_step(dec_hidden, enc_out, beam_seqs, beam_size)
    if all_done:
      break
  # Return the top results
  result=[]
  for top_beam in beam_seqs:
    kresult =''
    for pred_id,_ in top_beam:
      kresult += targ_lang.index_word[pred_id] + ' '
    result= result+[kresult]
  return result, scores_seq

In [ ]:
def translate_beam(sentence):
  result, score = evaluate_beam_search(sentence)
  return result, score